[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/differential_equations/08_odes_in_machine_learning/exercises.ipynb)

# Exercises: ODEs in Machine Learning

## Level 0 — Concept Check

### Problem L0.1: ResNets Are Euler Steps

**Problem Statement:** Show that the residual layer update $h_{l+1} = h_l + f(h_l, \theta_l)$ is exactly one forward-Euler step of an ordinary differential equation. Identify the ODE and the step size, and state what this correspondence predicts about the behavior of very deep residual networks.

**Intuition:** Adding a small correction to the current state each layer is the same arithmetic as advancing a trajectory by a small time step.

**Solution:**

The forward-Euler discretization of $\dot{h}(t) = F(h(t), t)$ with step $\Delta t$ is

$$
h_{n+1} = h_n + \Delta t \, F(h_n, t_n)
$$

Comparing term by term with $h_{l+1} = h_l + f(h_l, \theta_l)$, the two coincide with the identifications $\Delta t = 1$, $t_n = l$, and $F(h, l) = f(h, \theta_l)$, i.e., the layer index plays the role of time and the residual branch is the vector field.

Prediction for deep networks: as depth $L \to \infty$ with residual branches scaled like $1/L$, the network's forward pass converges to the solution of the ODE $\dot{h} = f(h, \theta(t))$; stability of forward propagation is governed by the eigenvalues of $I + \partial f/\partial h$, exactly the Euler stability condition, and the continuum limit is the Neural ODE.

$$
\boxed{h_{l+1} = h_l + f(h_l, \theta_l) \iff \text{forward Euler for } \dot{h} = f(h, \theta(t)) \text{ with } \Delta t = 1}
$$

> **Key Takeaway:** A ResNet is a fixed-step ODE solver whose vector field is learned; a Neural ODE is the same model with the solver made adaptive and the step size sent to zero.

---

### Problem L0.2: What the Adjoint State Is

**Problem Statement:** For a Neural ODE $\dot{h} = F(h, t, \theta)$ with scalar loss $\ell(h(T))$, define the adjoint state $a(t)$, state the ODE it satisfies and the direction of integration, give its dimension, and explain why the adjoint method uses memory independent of the number of forward solver steps.

**Solution:**

The adjoint state is the sensitivity of the loss to the state at time $t$:

$$
a(t) = \frac{\partial \ell}{\partial h(t)} \in \mathbb{R}^d
$$

It has the same dimension $d$ as the state $h(t)$, and satisfies the linear ODE

$$
\frac{da}{dt} = -\left( \frac{\partial F}{\partial h} \right)^{T} a(t), \qquad a(T) = \frac{\partial \ell}{\partial h(T)}
$$

integrated **backward** from $t = T$ to $t = 0$. Each evaluation of the right-hand side is one vector-Jacobian product, the same primitive used by ordinary backpropagation.

Memory independence: backprop through the solver must store (or recompute) every intermediate stage of every forward step, so memory grows with the number of function evaluations. The adjoint method instead reconstructs $h(t)$ by integrating the state equation backward together with $a(t)$ and the parameter-gradient accumulator; only the terminal state must be kept, so memory is $O(1)$ in the number of solver steps.

$$
\boxed{a(t) = \frac{\partial \ell}{\partial h(t)}, \qquad \dot{a} = -\left( \partial F/\partial h \right)^{T} a, \text{ solved from } T \text{ down to } 0}
$$

> **Key Takeaway:** The adjoint is continuous-time backpropagation: reverse-mode differentiation is itself a linear ODE, solved in reverse time.

---

### Problem L0.3: Why CNFs Need a Trace, Not a Determinant

**Problem Statement:** A discrete normalizing flow with invertible map $y = g(x)$ pays for the change-of-variables term $\log \lvert \det \left( \partial g/\partial x \right) \rvert$, which costs $O(d^3)$ in general. Explain why a continuous normalizing flow only needs the trace of the Jacobian of its vector field, state the relevant formula, and give the cost of estimating it with Hutchinson probes.

**Solution:**

In continuous time the flow map over an infinitesimal interval is $h \mapsto h + \varepsilon F(h, t)$, whose Jacobian is $I + \varepsilon \, \partial F/\partial h$. Its log-determinant expands as

$$
\log \det \left( I + \varepsilon \frac{\partial F}{\partial h} \right) = \varepsilon \operatorname{tr}\left( \frac{\partial F}{\partial h} \right) + O(\varepsilon^2)
$$

because to first order only the diagonal contributes. Summing over infinitesimal steps (Jacobi's formula / Liouville) turns the total log-determinant into a time integral of traces, giving the instantaneous change-of-variables formula

$$
\boxed{\frac{d}{dt} \log p_t(h(t)) = -\operatorname{tr}\left( \frac{\partial F}{\partial h}(h(t), t) \right)}
$$

Cost: the exact trace needs $d$ vector-Jacobian products ($O(d^2)$ for a dense Jacobian), but the Hutchinson estimator $\operatorname{tr}(J) = \mathbb{E}_v\left[ v^{T} J v \right]$ with a single random probe $v$ (Rademacher or Gaussian, identity covariance) needs one VJP per time step — $O(d)$ per evaluation, unbiased, with variance controlled by the number of probes.

> **Key Takeaway:** Determinants linearize to traces over infinitesimal steps; integrating a scalar trace is the entire likelihood bookkeeping of a CNF.

---

### Problem L0.4: Gradient Flow vs Gradient Descent

**Problem Statement:** State two qualitative properties that the gradient flow $\dot{\theta} = -\nabla L(\theta)$ always has but gradient descent $\theta_{k+1} = \theta_k - \eta \nabla L(\theta_k)$ can lose, and give the sharp stability condition on $\eta$ for GD on a quadratic loss with largest Hessian eigenvalue $L_{s}$.

**Solution:**

Property 1 — monotone descent. Along the flow,

$$
\frac{d}{dt} L(\theta(t)) = \nabla L^{T} \dot{\theta} = -\lVert \nabla L(\theta(t)) \rVert^2 \le 0
$$

so the continuous trajectory can never increase the loss. GD with too large a step can overshoot and increase $L$.

Property 2 — trajectories cannot oscillate or diverge on a convex quadratic: the flow $\dot{\theta} = -A\theta$ with $A \succeq 0$ contracts monotonically along every eigendirection. GD on the mode with curvature $L_{s}$ iterates $\theta_{k+1} = (1 - \eta L_{s})\theta_k$, which diverges in oscillating fashion when $\lvert 1 - \eta L_{s} \rvert \gt 1$.

Stability requires $-1 \le 1 - \eta L_{s} \le 1$ with the left inequality strict for convergence:

$$
\boxed{0 \lt \eta \lt \frac{2}{L_{s}}}
$$

This is precisely the absolute-stability interval of forward Euler applied to $\dot{y} = -L_{s} y$.

> **Key Takeaway:** Every pathology of large-step gradient descent is a numerical-stability phenomenon of forward Euler, not a property of the loss landscape alone.

---

## Level 1 — Foundation

### Problem L1.1: Exact Gradient Flow on a Quadratic Loss

**Problem Statement:** Solve the gradient flow $\dot{\theta} = -\nabla L(\theta)$ for the quadratic loss $L(\theta) = \tfrac{1}{2}\theta^{T} A \theta$ with $A$ symmetric positive definite, initial condition $\theta(0) = \theta_0$. Express the solution in the eigenbasis of $A$ and determine the asymptotic decay rate of $L(\theta(t))$.

**Solution:**

Since $\nabla L(\theta) = A\theta$, the flow is the linear system $\dot{\theta} = -A\theta$, whose solution (Topic 04) is the matrix exponential

$$
\theta(t) = e^{-At} \theta_0
$$

Diagonalize $A = Q \Lambda Q^{T}$ with orthonormal eigenvectors $q_i$ and eigenvalues $0 \lt \lambda_1 \le \dots \le \lambda_p$. Writing $\theta_0 = \sum_i c_i q_i$ with $c_i = q_i^{T}\theta_0$:

$$
\theta(t) = \sum_{i=1}^{p} c_i e^{-\lambda_i t} q_i
$$

Each Hessian eigendirection decays independently at rate $\lambda_i$: sharp directions are learned fast, flat directions slowly. The loss is

$$
L(\theta(t)) = \frac{1}{2} \sum_{i=1}^{p} \lambda_i c_i^2 e^{-2\lambda_i t}
$$

For generic $\theta_0$ (i.e., $c_1 \neq 0$) the slowest term dominates as $t \to \infty$, so $L$ decays like $e^{-2\lambda_1 t}$.

$$
\boxed{\theta(t) = e^{-At}\theta_0 = \sum_i c_i e^{-\lambda_i t} q_i, \qquad L(\theta(t)) \sim \tfrac{1}{2}\lambda_1 c_1^2 e^{-2\lambda_1 t}}
$$

> **Key Takeaway:** Training speed on a quadratic is a spectrum of independent exponential decays; the smallest Hessian eigenvalue sets the convergence bottleneck.

---

### Problem L1.2: One Euler Step vs the Exact Flow

**Problem Statement:** For the scalar test equation $\dot{y} = -\lambda y$ with $\lambda \gt 0$ and $y(0) = y_0$, compute the exact solution after one step of size $h$ and the forward-Euler approximation, and derive the leading term of the local truncation error. For which $h$ does Euler remain stable?

**Solution:**

Exact: $y(h) = y_0 e^{-\lambda h}$. Euler: $y_1 = y_0 (1 - \lambda h)$.

Expand the exponential:

$$
e^{-\lambda h} = 1 - \lambda h + \frac{\lambda^2 h^2}{2} - \frac{\lambda^3 h^3}{6} + \dots
$$

so the local error after one step is

$$
y(h) - y_1 = y_0 \left( e^{-\lambda h} - 1 + \lambda h \right) = y_0 \left( \frac{\lambda^2 h^2}{2} - \frac{\lambda^3 h^3}{6} + \dots \right)
$$

with leading term $\tfrac{1}{2}\lambda^2 h^2 y_0$ — first-order accuracy globally (error $O(h)$ over a fixed horizon after $\sim 1/h$ steps).

Stability: the Euler amplification factor is $1 - \lambda h$; iterates decay in magnitude iff $\lvert 1 - \lambda h \rvert \lt 1$, i.e., $0 \lt h \lt 2/\lambda$.

$$
\boxed{y(h) - y_1 = \frac{\lambda^2 h^2}{2} y_0 + O(h^3), \qquad \text{stable iff } h \lt \frac{2}{\lambda}}
$$

> **Key Takeaway:** The $O(h^2)$ per-step error and the $2/\lambda$ stability ceiling are the two numbers that translate directly into ResNet depth scaling and learning-rate limits.

---

### Problem L1.3: Classifying Heavy-Ball Damping Regimes

**Problem Statement:** For the momentum ODE $\ddot{\theta} + \gamma\dot{\theta} + 4\theta = 0$ (curvature $\lambda = 4$), find the characteristic roots and classify the dynamics for $\gamma = 2$, $\gamma = 4$, and $\gamma = 5$. Which value gives the fastest asymptotic decay?

**Solution:**

The characteristic equation (Topic 03) is $r^2 + \gamma r + 4 = 0$ with discriminant $\gamma^2 - 16$.

Case $\gamma = 2$: $r = \frac{-2 \pm \sqrt{4 - 16}}{2} = -1 \pm i\sqrt{3}$ — complex roots, **underdamped**: oscillation with envelope $e^{-t}$ (decay rate 1).

Case $\gamma = 4$: $r^2 + 4r + 4 = (r + 2)^2 = 0$, double root $r = -2$ — **critically damped**: $\theta(t) = (c_1 + c_2 t)e^{-2t}$, decay rate 2, no oscillation.

Case $\gamma = 5$: $r^2 + 5r + 4 = (r + 1)(r + 4) = 0$, roots $-1$ and $-4$ — **overdamped**: the slow root $-1$ dominates, decay rate 1.

Comparing asymptotic decay rates $1$, $2$, $1$: the critical value wins, consistent with the general optimum $\gamma^{\ast} = 2\sqrt{\lambda} = 4$.

$$
\boxed{\gamma = 2: \text{ underdamped, rate } 1; \quad \gamma = 4: \text{ critical, rate } 2; \quad \gamma = 5: \text{ overdamped, rate } 1}
$$

> **Key Takeaway:** Momentum is a damping knob: too little oscillates, too much crawls, and the optimum sits exactly at critical damping $\gamma = 2\sqrt{\lambda}$.

---

### Problem L1.4: The Adjoint Method on a Scalar Linear Neural ODE

**Problem Statement:** Consider the scalar Neural ODE $\dot{h} = w h$ on $[0, 1]$ with parameter $w$, initial state $h(0) = h_0$, and loss $\ell = \tfrac{1}{2} h(1)^2$. Compute $d\ell/dw$ (i) directly from the closed-form solution and (ii) via the adjoint method, and verify that the two agree.

**Solution:**

**(i) Direct.** The solution is $h(t) = h_0 e^{wt}$, so $h(1) = h_0 e^{w}$ and

$$
\frac{d\ell}{dw} = h(1) \cdot \frac{dh(1)}{dw} = h_0 e^{w} \cdot h_0 e^{w} = h_0^2 e^{2w}
$$

**(ii) Adjoint.** Here $\partial F/\partial h = w$ and $\partial F/\partial w = h$. The adjoint $\lambda(t)$ satisfies

$$
\dot{\lambda} = -w \lambda, \qquad \lambda(1) = \frac{\partial \ell}{\partial h(1)} = h(1)
$$

Solving backward: $\lambda(t) = h(1) e^{w(1 - t)}$. The parameter gradient is

$$
\frac{d\ell}{dw} = \int_0^1 \lambda(t) \, \frac{\partial F}{\partial w} \, dt = \int_0^1 h(1) e^{w(1-t)} \cdot h_0 e^{wt} \, dt = h(1) h_0 e^{w} \int_0^1 dt = h_0 e^{w} \cdot h_0 e^{w}
$$

which equals $h_0^2 e^{2w}$, matching (i).

$$
\boxed{\frac{d\ell}{dw} = h_0^2 e^{2w} \quad \text{by both methods}}
$$

> **Key Takeaway:** The adjoint integral silently reproduces the chain rule: the factor $\lambda(t)$ carries the downstream sensitivity while $\partial F/\partial w$ carries the local one.

---

### Problem L1.5: Trace Formula for a Linear Vector Field

**Problem Statement:** Let the CNF dynamics be linear, $F(h) = W h$ with constant $W \in \mathbb{R}^{d \times d}$. Integrate the instantaneous change-of-variables formula over $[0, t]$, and verify the result independently using the flow map $h(t) = e^{Wt} h(0)$ and the identity $\det e^{Wt} = e^{t \operatorname{tr} W}$.

**Solution:**

**Via the trace formula.** Here $\partial F/\partial h = W$, so along any trajectory

$$
\frac{d}{dt} \log p_t(h(t)) = -\operatorname{tr}(W)
$$

The right side is constant, so integrating from $0$ to $t$:

$$
\log p_t(h(t)) = \log p_0(h(0)) - t \operatorname{tr}(W)
$$

**Verification via determinants.** The flow map is the linear map $e^{Wt}$, whose Jacobian is $e^{Wt}$ itself. The discrete change-of-variables formula gives

$$
\log p_t(h(t)) = \log p_0(h(0)) - \log \det e^{Wt} = \log p_0(h(0)) - t \operatorname{tr}(W)
$$

using Liouville's identity $\det e^{Wt} = e^{t \operatorname{tr} W}$ (Topic 04). The two computations agree exactly.

$$
\boxed{\log p_t(h(t)) = \log p_0(h(0)) - t \operatorname{tr}(W)}
$$

Volume interpretation: if $\operatorname{tr}(W) \gt 0$ the flow expands volume and densities dilute; if $\operatorname{tr}(W) \lt 0$ it contracts volume and densities concentrate.

> **Key Takeaway:** For linear dynamics the CNF likelihood correction is exactly $t \operatorname{tr}(W)$ — the trace formula is Liouville's theorem in Lagrangian coordinates.

---

### Problem L1.6: Zero-Order-Hold Discretization of a Scalar System

**Problem Statement:** Derive the exact discretization of the scalar state-space model $\dot{x} = a x + b u$, $a \neq 0$, under a zero-order hold: the input is constant, $u(t) = u_k$, on each interval $[k\Delta, (k+1)\Delta)$. Express $x_{k+1}$ in terms of $x_k$ and $u_k$.

**Solution:**

On one hold interval the equation is linear with constant forcing, so Duhamel's formula (Topic 04) applies with initial state $x_k$ at local time $0$:

$$
x_{k+1} = e^{a\Delta} x_k + \int_0^{\Delta} e^{a(\Delta - s)} b \, u_k \, ds
$$

The input is constant, so the integral is elementary:

$$
\int_0^{\Delta} e^{a(\Delta - s)} \, ds = \left[ -\frac{1}{a} e^{a(\Delta - s)} \right]_0^{\Delta} = \frac{e^{a\Delta} - 1}{a}
$$

Therefore

$$
\boxed{x_{k+1} = e^{a\Delta} x_k + \frac{b}{a} \left( e^{a\Delta} - 1 \right) u_k}
$$

Sanity checks: as $\Delta \to 0$, $e^{a\Delta} \approx 1 + a\Delta$ recovers Euler $x_{k+1} \approx x_k + \Delta(a x_k + b u_k)$; the matrix version $\bar{A} = e^{A\Delta}$, $\bar{B} = A^{-1}(e^{A\Delta} - I)B$ is exactly the discretization used by S4/Mamba layers.

> **Key Takeaway:** ZOH discretization is exact, not approximate: state-space sequence layers advance their hidden state with a true matrix exponential, inheriting its stability guarantees.

---

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: Tuning Momentum for a Hessian Spectrum

**Problem Statement:** A loss surface near its minimum has Hessian eigenvalues filling $[\mu, L_{s}] = [1, 100]$. Using the heavy-ball ODE $\ddot{\theta} + \gamma\dot{\theta} + \lambda\theta = 0$ per mode, determine the single damping $\gamma$ that maximizes the worst-case asymptotic decay rate over all modes, compute that rate, and explain where the celebrated $\sqrt{\kappa}$ acceleration over gradient descent actually comes from.

**Solution:**

Per-mode rates as a function of $\gamma$: a mode with curvature $\lambda$ decays at rate $\gamma/2$ if underdamped ($\gamma \le 2\sqrt{\lambda}$) and at rate $\frac{\gamma - \sqrt{\gamma^2 - 4\lambda}}{2}$ if overdamped, which is decreasing in $\gamma$ and less than $\sqrt{\lambda}$.

The worst mode is always the flattest one, $\lambda = \mu = 1$: for $\gamma \le 2\sqrt{\mu}$ every mode is (at most critically) underdamped and all decay at the common envelope rate $\gamma/2$, which grows with $\gamma$; for $\gamma \gt 2\sqrt{\mu}$ the $\mu$-mode becomes overdamped and its rate falls below $\sqrt{\mu}$. The optimum is therefore the critical damping of the slowest mode:

$$
\gamma^{\ast} = 2\sqrt{\mu} = 2, \qquad \text{worst-case rate} = \sqrt{\mu} = 1
$$

At this $\gamma$, the $\lambda = 100$ mode is strongly underdamped (it rings at frequency $\approx \sqrt{100 - 1} \approx 9.95$) but its envelope still decays at rate 1 — ringing of sharp modes is the accepted price for speed on flat ones.

Where acceleration lives: plain gradient flow decays the $\mu$-mode at rate $\mu$; the tuned heavy-ball flow decays it at rate $\sqrt{\mu}$. After discretization with the step allowed by the sharpest curvature, these become per-step factors $\approx 1 - \mu/L_{s} = 1 - 1/\kappa$ for GD versus $\approx 1 - \sqrt{\mu/L_{s}} = 1 - 1/\sqrt{\kappa}$ for momentum, with condition number $\kappa = L_{s}/\mu = 100$: about $10\times$ fewer iterations here.

$$
\boxed{\gamma^{\ast} = 2\sqrt{\mu} = 2, \qquad \text{rate } \sqrt{\mu}, \qquad \text{iteration gain } \approx \sqrt{\kappa} = 10}
$$

> **Key Takeaway:** Momentum should critically damp the flattest Hessian direction; the $\sqrt{\kappa}$ speedup is a damped-oscillator resonance fact, visible already in continuous time.

---

### Problem L2.2: NFE Budgets — Neural ODE vs ResNet Depth

**Problem Statement:** A ResNet with $D = 32$ residual blocks costs 32 evaluations of $f$ per forward pass and integrates its feature flow with accuracy $O(h)$ per unit time. A Neural ODE over $t \in [0, 1]$ using fixed-step RK4 with $N$ steps costs $4N$ evaluations. (a) How many RK4 steps match the ResNet's evaluation budget? (b) Compare the integration errors at equal budget, assuming error constants of order 1. (c) Why can an adaptive solver beat both?

**Solution:**

**(a)** Equal budget requires $4N = 32$, so $N = 8$ RK4 steps of size $h = 1/8$.

**(b)** Global error orders (Topic on numerical methods): Euler-like ResNet integration error scales as $O(h_{\text{res}})$ with $h_{\text{res}} = 1/32$, giving error $\approx 1/32 \approx 3.1 \times 10^{-2}$. RK4 error scales as $O(h^4)$ with $h = 1/8$:

$$
\left( \tfrac{1}{8} \right)^4 = \frac{1}{4096} \approx 2.4 \times 10^{-4}
$$

At the same evaluation budget, RK4 tracks the continuous flow roughly two orders of magnitude more accurately — equivalently, it needs far fewer evaluations for the same fidelity.

**(c)** An adaptive solver estimates local error and concentrates steps where the dynamics are fast, so smooth inputs get cheap passes (low NFE) while hard inputs automatically get "deeper" computation. NFE thereby becomes an input-dependent, learned notion of depth, and can drop further with regularizers that straighten the flow.

$$
\boxed{N = 8 \text{ RK4 steps}; \quad \text{error} \approx 2.4 \times 10^{-4} \text{ vs } 3.1 \times 10^{-2} \text{ at equal budget}}
$$

> **Key Takeaway:** Depth in a Neural ODE is a solver decision: higher-order and adaptive integrators buy accuracy per evaluation that a fixed-depth ResNet cannot match.

---

### Problem L2.3: A Latent ODE for Irregularly-Sampled Vitals

**Problem Statement:** A patient's latent health state follows $\dot{z} = -z/\tau$ with $\tau = 2$ hours between measurements; at each measurement the encoder applies a jump update. Observations arrive at irregular times $t_0 = 0$, $t_1 = 0.5$, $t_2 = 2.0$ hours. Starting from $z(t_0^{+}) = 4$: (a) derive the general inter-observation propagation rule; (b) compute the model state just before $t_1$ and, after the jump sets $z(t_1^{+}) = 3.5$, just before $t_2$; (c) explain why a discrete RNN with a fixed step cannot represent this correctly.

**Solution:**

**(a)** Between observations the ODE is linear (Topic 01); solving from $t_i$ over a gap $\Delta_i = t_{i+1} - t_i$:

$$
z(t_{i+1}^{-}) = z(t_i^{+}) \, e^{-\Delta_i/\tau}
$$

The decay applied between updates depends explicitly on the elapsed time $\Delta_i$ — this is the defining feature of ODE-RNN/latent-ODE models.

**(b)** Gap 1: $\Delta_0 = 0.5$, so

$$
z(t_1^{-}) = 4 e^{-0.5/2} = 4 e^{-0.25} \approx 3.115
$$

After the encoder jump, $z(t_1^{+}) = 3.5$. Gap 2: $\Delta_1 = 1.5$, so

$$
z(t_2^{-}) = 3.5 e^{-1.5/2} = 3.5 e^{-0.75} \approx 1.653
$$

**(c)** A standard RNN applies the same transition map once per observation regardless of $\Delta_i$, implicitly assuming uniform spacing: it would decay the state equally across the $0.5$-hour and $1.5$-hour gaps. The latent ODE instead exponentiates the true elapsed time, so its predictions remain calibrated under arbitrary sampling patterns and can be queried at any time between observations.

$$
\boxed{z(t_{i+1}^{-}) = z(t_i^{+}) e^{-\Delta_i/\tau}; \quad z(t_1^{-}) \approx 3.115, \quad z(t_2^{-}) \approx 1.653}
$$

> **Key Takeaway:** Continuous-time latent dynamics make elapsed time a first-class input; the ODE solve between events is what fixed-step RNNs are missing.

---

### Problem L2.4: Probability Flow ODE for Gaussian Data

**Problem Statement:** For the VP diffusion with constant $\beta$, initial data $x_0 \sim \mathcal{N}(0, v_0)$ with $v_0 = 4$, the marginals stay Gaussian: $p_t = \mathcal{N}(0, v(t))$ with $v(t) = 1 + (v_0 - 1)e^{-\beta t}$. The probability flow ODE is $\dot{x} = -\tfrac{\beta}{2} x - \tfrac{\beta}{2} \nabla_x \log p_t(x)$. Show that $x(t) = x_0 \sqrt{v(t)/v_0}$ solves it, and interpret the result.

**Solution:**

For a centered Gaussian, $\log p_t(x) = -x^2/(2v(t)) + \text{const}$, so the score is $\nabla_x \log p_t(x) = -x/v(t)$ and the ODE becomes

$$
\dot{x} = -\frac{\beta}{2} x + \frac{\beta}{2} \frac{x}{v(t)} = \frac{\beta \left( 1 - v(t) \right)}{2 v(t)} \, x
$$

Now test the candidate $x(t) = x_0 \sqrt{v(t)/v_0}$. Differentiating,

$$
\dot{x} = x_0 \frac{v'(t)}{2\sqrt{v_0 \, v(t)}} = \frac{v'(t)}{2 v(t)} \, x(t)
$$

The variance ODE from the OU moment computation is $v' = \beta(1 - v)$, so

$$
\dot{x} = \frac{\beta\left( 1 - v(t) \right)}{2 v(t)} \, x(t)
$$

which is exactly the right-hand side. Hence the candidate is the solution through $x(0) = x_0$.

Interpretation: with $v_0 = 4$, $v(t) = 1 + 3e^{-\beta t}$ shrinks from 4 toward 1, and every sample moves along $x(t) = x_0\sqrt{v(t)}/2$ — a deterministic, monotone rescaling that preserves quantiles: the sample at the 90th percentile of the data distribution flows to the 90th percentile of the prior. Reversing time gives DDIM-style deterministic sampling.

$$
\boxed{x(t) = x_0 \sqrt{\frac{v(t)}{v_0}}, \qquad v(t) = 1 + 3e^{-\beta t}}
$$

> **Key Takeaway:** For Gaussian marginals the probability flow ODE is a pure variance reparameterization — deterministic diffusion sampling is quantile transport along the noise schedule.

---

### Problem L2.5: The LSTM Forget Gate as a Leaky Integrator

**Problem Statement:** With no new input, an LSTM cell state obeys $c_{n+1} = f \cdot c_n$ for forget-gate value $f \in (0, 1)$, applied once per step of duration $\Delta$. Model this as sampling the leaky integrator ODE $\dot{c} = -c/\tau$. (a) Express the effective time constant $\tau$ in terms of $f$ and $\Delta$. (b) Evaluate $\tau$ (in steps, $\Delta = 1$) for $f = 0.9$ and $f = 0.99$. (c) Explain what this says about learning long-range dependencies.

**Solution:**

**(a)** The ODE decays as $c(t + \Delta) = c(t) e^{-\Delta/\tau}$. Matching one LSTM step $c_{n+1} = f c_n$ gives $e^{-\Delta/\tau} = f$, hence

$$
\tau = -\frac{\Delta}{\ln f}
$$

**(b)** With $\Delta = 1$:

$$
f = 0.9: \quad \tau = -\frac{1}{\ln 0.9} \approx \frac{1}{0.10536} \approx 9.49 \text{ steps}
$$

$$
f = 0.99: \quad \tau = -\frac{1}{\ln 0.99} \approx \frac{1}{0.01005} \approx 99.5 \text{ steps}
$$

**(c)** The memory horizon is exponentially sensitive to the gate: pushing $f$ from 0.9 to 0.99 stretches the time constant by roughly $10\times$. Because $f = \sigma(\cdot)$ is trainable per unit and per step, the LSTM learns a bank of leaky integrators with heterogeneous timescales — the gating mechanism is timescale selection in a sampled ODE, which is why initializing forget-gate biases high (chrono initialization) helps long-range tasks.

$$
\boxed{\tau = -\frac{\Delta}{\ln f}; \qquad \tau(0.9) \approx 9.5, \quad \tau(0.99) \approx 99.5 \text{ steps}}
$$

> **Key Takeaway:** A forget gate is $e^{-\Delta/\tau}$ in disguise; gated RNNs place their dynamics' eigenvalues just inside the unit circle to keep gradients alive over long horizons.

---

### Problem L2.6: Computing an S4-Style Convolution Kernel

**Problem Statement:** A continuous state-space layer has

$$
A = \begin{pmatrix} -1 & 1 \\ 0 & -1 \end{pmatrix}, \qquad B = \begin{pmatrix} 0 \\ 1 \end{pmatrix}, \qquad C = \begin{pmatrix} 1 & 0 \end{pmatrix}
$$

Compute the impulse-response kernel $K(t) = C e^{At} B$ that the layer convolves its input with, and identify its transfer function (Topic 06 link).

**Solution:**

Write $A = -I + N$ with the nilpotent $N = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$, $N^2 = 0$. Since $-I$ and $N$ commute (Topic 04),

$$
e^{At} = e^{-t} e^{Nt} = e^{-t} \left( I + Nt \right) = e^{-t} \begin{pmatrix} 1 & t \\ 0 & 1 \end{pmatrix}
$$

Then

$$
K(t) = C e^{At} B = e^{-t} \begin{pmatrix} 1 & 0 \end{pmatrix} \begin{pmatrix} 1 & t \\ 0 & 1 \end{pmatrix} \begin{pmatrix} 0 \\ 1 \end{pmatrix} = e^{-t} \begin{pmatrix} 1 & t \end{pmatrix} \begin{pmatrix} 0 \\ 1 \end{pmatrix} = t e^{-t}
$$

The layer's output is $y(t) = (K \ast u)(t)$: a smooth, causal kernel that rises, peaks at $t = 1$, and decays exponentially — a learned, stable memory profile. Its Laplace transform is

$$
\mathcal{L}\{ t e^{-t} \}(s) = \frac{1}{(s + 1)^2}
$$

a double pole at $s = -1$, i.e., the transfer function of a critically damped second-order filter; stability follows from the pole lying in the left half-plane.

$$
\boxed{K(t) = t e^{-t}, \qquad H(s) = \frac{1}{(s + 1)^2}}
$$

> **Key Takeaway:** SSM sequence layers are ODE-defined convolutions: the kernel is $C e^{At} B$, its shape is the layer's memory, and its poles (eigenvalues of $A$) certify stability.

---

## Level 3 — Challenge

### Problem L3.1: The $O(1/t^2)$ Rate of the Nesterov ODE

**Problem Statement:** Let $f$ be convex and differentiable with minimizer $x^{\ast}$, and let $X(t)$ solve the Nesterov ODE

$$
\ddot{X} + \frac{3}{t}\dot{X} + \nabla f(X) = 0, \qquad X(0) = x_0, \quad \dot{X}(0) = 0
$$

Prove that $f(X(t)) - f^{\ast} \le \dfrac{2\lVert x_0 - x^{\ast} \rVert^2}{t^2}$ using the Lyapunov function

$$
\mathcal{E}(t) = t^2\left( f(X(t)) - f^{\ast} \right) + 2\left\lVert X(t) + \tfrac{t}{2}\dot{X}(t) - x^{\ast} \right\rVert^2
$$

**Solution:**

**Step 1 — Differentiate the first term.**

$$
\frac{d}{dt}\left[ t^2 (f(X) - f^{\ast}) \right] = 2t\left( f(X) - f^{\ast} \right) + t^2 \langle \nabla f(X), \dot{X} \rangle
$$

**Step 2 — Differentiate the second term.** Let $u(t) = X + \tfrac{t}{2}\dot{X} - x^{\ast}$, so $\frac{d}{dt} 2\lVert u \rVert^2 = 4\langle u, \dot{u} \rangle$ with

$$
\dot{u} = \dot{X} + \tfrac{1}{2}\dot{X} + \tfrac{t}{2}\ddot{X} = \tfrac{3}{2}\dot{X} + \tfrac{t}{2}\ddot{X}
$$

The ODE gives $\ddot{X} = -\tfrac{3}{t}\dot{X} - \nabla f(X)$, hence

$$
\dot{u} = \tfrac{3}{2}\dot{X} - \tfrac{3}{2}\dot{X} - \tfrac{t}{2}\nabla f(X) = -\tfrac{t}{2}\nabla f(X)
$$

— the damping coefficient $3/t$ is precisely what makes the velocity terms cancel.

**Step 3 — Combine.**

$$
\dot{\mathcal{E}} = 2t(f(X) - f^{\ast}) + t^2\langle \nabla f, \dot{X} \rangle - 2t\left\langle X + \tfrac{t}{2}\dot{X} - x^{\ast}, \nabla f \right\rangle
$$

Expanding the last inner product, the $t^2\langle \nabla f, \dot{X} \rangle$ terms cancel:

$$
\dot{\mathcal{E}} = 2t\left( f(X) - f^{\ast} \right) - 2t\left\langle \nabla f(X), X - x^{\ast} \right\rangle
$$

**Step 4 — Convexity.** For convex $f$, $f(x^{\ast}) \ge f(X) + \langle \nabla f(X), x^{\ast} - X \rangle$, i.e., $\langle \nabla f(X), X - x^{\ast} \rangle \ge f(X) - f^{\ast}$. Therefore $\dot{\mathcal{E}} \le 2t(f - f^{\ast}) - 2t(f - f^{\ast}) = 0$: $\mathcal{E}$ is non-increasing.

**Step 5 — Conclude.** $\mathcal{E}(0) = 0 + 2\lVert x_0 - x^{\ast} \rVert^2$, and since both terms of $\mathcal{E}$ are non-negative,

$$
t^2 \left( f(X(t)) - f^{\ast} \right) \le \mathcal{E}(t) \le \mathcal{E}(0) = 2\lVert x_0 - x^{\ast} \rVert^2
$$

$$
\boxed{f(X(t)) - f^{\ast} \le \frac{2\lVert x_0 - x^{\ast} \rVert^2}{t^2}}
$$

> **Key Takeaway:** Acceleration is a Lyapunov phenomenon: the anomalous $3/t$ damping is exactly the coefficient for which the energy $\mathcal{E}$ dissipates, forcing the $O(1/t^2)$ rate that no constant-damping flow achieves for general convex $f$.

---

### Problem L3.2: Adjoint Method as the Continuum Limit of Backpropagation

**Problem Statement:** Let the Neural ODE vector field be piecewise linear in time: $F(h, t) = W_k h$ for $t \in [t_k, t_{k+1})$, $k = 0, \dots, K-1$, a continuous-time model of a deep linear network. (a) Compute the forward flow map $\Phi$ over $[t_0, t_K]$. (b) Solve the adjoint equation across the intervals and show $a(t_0) = \left( \partial \Phi/\partial h_0 \right)^{T} a(t_K)$, the backpropagation chain rule. (c) Conclude the general principle.

**Solution:**

**(a) Forward.** On each interval the system is linear with constant matrix, so with $\delta_k = t_{k+1} - t_k$ (Topic 04):

$$
h(t_K) = e^{W_{K-1}\delta_{K-1}} \cdots e^{W_1 \delta_1} e^{W_0 \delta_0} \, h(t_0) =: \Phi\, h(t_0)
$$

— an ordered product of layer maps, rightmost acting first, exactly a deep linear network with layers $e^{W_k \delta_k}$.

**(b) Backward.** The adjoint equation is $\dot{a} = -W_k^{T} a$ on $[t_k, t_{k+1})$. Solving backward on one interval from the terminal value $a(t_{k+1})$:

$$
a(t_k) = e^{W_k^{T} \delta_k} \, a(t_{k+1})
$$

(the sign flips because we integrate in reverse time). Chaining over all intervals from $k = K-1$ down to $0$:

$$
a(t_0) = e^{W_0^{T}\delta_0} e^{W_1^{T}\delta_1} \cdots e^{W_{K-1}^{T}\delta_{K-1}} \, a(t_K) = \left( e^{W_{K-1}\delta_{K-1}} \cdots e^{W_0\delta_0} \right)^{T} a(t_K) = \Phi^{T} a(t_K)
$$

using $\left( M_1 M_2 \cdots M_K \right)^{T} = M_K^{T} \cdots M_2^{T} M_1^{T}$. This is precisely reverse-mode differentiation: transposed layer Jacobians applied in reverse order.

**(c) Principle.** For general $F$, the forward sensitivity solves the variational equation $\dot{\Psi} = (\partial F/\partial h)\Psi$ and the adjoint solves its transpose backward, so the identity $a(t_0) = \Psi(t_K)^{T} a(t_K)$ holds always; backprop through any discretization converges to the adjoint solution as steps refine.

$$
\boxed{a(t_0) = \left( \frac{\partial h(t_K)}{\partial h(t_0)} \right)^{T} a(t_K) \quad \text{— backprop is the adjoint ODE, exactly}}
$$

> **Key Takeaway:** Backpropagation and the adjoint method are one algorithm at two resolutions: transposed-Jacobian products are the discrete samples of a single reverse-time linear ODE.

---

### Problem L3.3: An Expressiveness No-Go and Its Augmentation Fix

**Problem Statement:** (a) Prove that no Neural ODE on $\mathbb{R}$ (state dimension 1, $F$ continuous in $t$ and Lipschitz in $x$) has time-$T$ flow map $\Phi_T(x) = -x$. (b) Exhibit an explicit augmented system on $\mathbb{R}^2$ whose flow restricted to the axis $(x, 0)$ realizes $x \mapsto -x$ in the first coordinate.

**Solution:**

**(a) 1D flows preserve order.** Let $x_1 \lt x_2$ and let $h_1(t), h_2(t)$ be the corresponding solutions. Suppose they ever met: $h_1(t^{\ast}) = h_2(t^{\ast})$ for some $t^{\ast} \in [0, T]$. Both curves then solve the same IVP at $t^{\ast}$, and by Picard–Lindelöf uniqueness (Topic 02, applied forward and backward from $t^{\ast}$) they coincide everywhere — contradicting $h_1(0) \neq h_2(0)$. Hence trajectories never cross, and by continuity the initial ordering persists:

$$
x_1 \lt x_2 \implies \Phi_T(x_1) \lt \Phi_T(x_2)
$$

so every 1D Neural ODE flow is strictly increasing. But $\Phi_T(x) = -x$ is strictly decreasing (e.g., it maps $-1 \lt 1$ to $1 \gt -1$), a contradiction. No such vector field exists.

**(b) Augmentation.** Lift to $\mathbb{R}^2$ and rotate. Take the linear field with

$$
\dot{z} = A z, \qquad A = \pi \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}, \qquad T = 1
$$

Then $e^{AT}$ is the rotation by angle $\pi$ (Topic 04):

$$
e^{A} = \begin{pmatrix} \cos\pi & -\sin\pi \\ \sin\pi & \cos\pi \end{pmatrix} = -I
$$

Starting from the augmented point $(x, 0)$, the flow ends at $(-x, 0)$; projecting to the first coordinate realizes $x \mapsto -x$. The trajectory leaves the axis, travels through the extra dimension, and returns — exactly what is impossible inside $\mathbb{R}$.

$$
\boxed{\text{1D flows are strictly increasing, so } x \mapsto -x \text{ is unreachable; one extra dimension and a } \pi\text{-rotation realize it}}
$$

> **Key Takeaway:** Uniqueness of solutions makes Neural ODE flows orientation-preserving homeomorphisms; augmented dimensions restore expressiveness by letting trajectories go around obstacles rather than through them.

---

### Problem L3.4: Exact Probability Flow with a Nonzero Mean

**Problem Statement:** For the VP diffusion with constant $\beta$ and Gaussian initial data $x_0 \sim \mathcal{N}(m_0, v_0)$, the marginals are $p_t = \mathcal{N}(m(t), v(t))$ with $m(t) = m_0 e^{-\beta t/2}$ and $v(t) = 1 + (v_0 - 1)e^{-\beta t}$. Show that the probability flow ODE $\dot{x} = -\tfrac{\beta}{2}x - \tfrac{\beta}{2}\nabla_x \log p_t(x)$ is solved by

$$
x(t) = m(t) + \sqrt{\frac{v(t)}{v_0}} \left( x_0 - m_0 \right)
$$

and interpret the structure of this map for deterministic diffusion samplers.

**Solution:**

**Step 1 — Score of a Gaussian.** $\log p_t(x) = -\frac{(x - m(t))^2}{2v(t)} + \text{const}$, so $\nabla_x \log p_t(x) = -\frac{x - m(t)}{v(t)}$, and the ODE reads

$$
\dot{x} = -\frac{\beta}{2} x + \frac{\beta}{2} \cdot \frac{x - m(t)}{v(t)}
$$

**Step 2 — Differentiate the candidate.** With $x(t) - m(t) = \sqrt{v(t)/v_0}\,(x_0 - m_0)$,

$$
\dot{x} = m'(t) + \frac{v'(t)}{2\sqrt{v_0 v(t)}}\left( x_0 - m_0 \right) = m'(t) + \frac{v'(t)}{2v(t)}\left( x(t) - m(t) \right)
$$

**Step 3 — Match both parts.** The moment ODEs are $m' = -\tfrac{\beta}{2}m$ and $v' = \beta(1 - v)$. Substituting these:

$$
\dot{x} = -\frac{\beta}{2}m(t) + \frac{\beta\left( 1 - v(t) \right)}{2v(t)}\left( x(t) - m(t) \right)
$$

Meanwhile the right-hand side of the ODE from Step 1 decomposes as

$$
-\frac{\beta}{2}\left[ m(t) + \left( x - m(t) \right) \right] + \frac{\beta}{2}\frac{x - m(t)}{v(t)} = -\frac{\beta}{2}m(t) + \frac{\beta(1 - v(t))}{2v(t)}\left( x - m(t) \right)
$$

The two expressions agree identically, and the candidate satisfies $x(0) = m_0 + (x_0 - m_0) = x_0$. By uniqueness (the right side is Lipschitz in $x$), it is the solution.

**Step 4 — Interpretation.** The flow map at each time is **affine**: recenter by the decaying mean, rescale fluctuations by $\sqrt{v(t)/v_0}$. It transports each quantile of $p_0$ to the same quantile of $p_t$, is invertible in closed form, and commutes with time reversal — which is why DDIM sampling (integrating this ODE backward from the prior) is deterministic, invertible "noise coordinates" for data, enabling exact reconstruction and semantic interpolation.

$$
\boxed{x(t) = m(t) + \sqrt{\frac{v(t)}{v_0}}\left( x_0 - m_0 \right) \quad \text{solves the probability flow ODE}}
$$

> **Key Takeaway:** On Gaussians the probability flow is an affine quantile transport built from the OU moment ODEs — the cleanest window into why deterministic diffusion samplers work.

---